# PGCN-EEG revision analyses (no training)

Runs everything for the revision in order:

1. **Label check** against the official DEAP ratings
2. **Re-scoring** of all 438 checkpoints with the training batch sizes (32 within, 16 LOSO)
3. **Statistics**: model vs majority baseline, fusion vs EEG-only (matched s01-s22), ablations; Wilcoxon + BH
4. **ViT probes**: identity, subject-centred affect probes, permutation nulls, session-drift check
5. **Test-time video shuffle** on the within-subject fusion models

**Before running**
- Settings → Accelerator: **GPU**. Settings → Internet: **On** (for the PyG install).
- Add these inputs: `pgcn-all-results`, `pgcn-deap-preprocessed`, `pgcn-deap-fusion-preprocessed`,
  `pgcn-eeg-code`, the DEAP `.dat` files (`data-preprocessed-python`), and a dataset containing the
  official **`participant_ratings.csv`** (from DEAP's `metadata_csv.zip`).
- Then **Save Version → Save & Run All**. Total time is roughly 30-60 minutes.

In [ ]:
# 1. Install PyTorch Geometric
import subprocess, sys, torch
assert torch.cuda.is_available(), "Turn on the GPU: Settings -> Accelerator -> GPU"
tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".", "")
url = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyg_lib", "torch_scatter", "torch_sparse", "-f", url], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True)
import torch_geometric; print("PyG", torch_geometric.__version__)

## 2. Write the analysis scripts

In [ ]:
%%writefile /kaggle/working/reevaluate_checkpoints.py
"""
reevaluate_checkpoints.py
=========================
Re-score the saved best.pt checkpoints WITHOUT retraining.

Why: pgcn_experiment.evaluate() reports f1_score(average="binary"), which is the
F1 of the positive class only, not macro-F1. A model that predicts "high" for
every window scores 2p/(1+p) on that metric (about 0.69 to 0.92 for DEAP label
proportions), so collapse to the majority class looks like good performance.

For every subject or fold this script reports, per task:
  * f1_pos     : positive-class F1 (the metric used in the submitted paper)
  * macro_f1   : macro-averaged F1 over the classes
  * bal_acc    : balanced accuracy (chance = 0.5 binary, 0.25 quadrant)
  * acc        : plain accuracy
  * pred_pos   : fraction of windows/trials predicted as class 1 (collapse check)
at three levels:
  * window     : as in the paper
  * trial_mean : mean of the 29 window probabilities per trial, then argmax
  * trial_vote : majority vote over the 29 window predictions per trial
and a majority-class baseline (majority label of the training trials the model
saw: the subject's own 28 training trials within-subject, the training trials
of the other 31 subjects under LOSO), scored the same way.

It also saves the per-window probabilities to an .npz file so any later
statistic can be recomputed without touching the GPU again.

Usage (Kaggle, after copying the repo .py files to /kaggle/working):

  python reevaluate_checkpoints.py \
      --data_dir /kaggle/input/.../pgcn-deap-preprocessed \
      --ckpt_dir /kaggle/input/.../results_eeg/loso_none \
      --mode loso --fusion none --ablation none \
      --out_prefix /kaggle/working/reeval/loso_none

Run once per configuration (within/loso x none/early/late, plus each ablation).
"""

import argparse
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch_geometric.loader import DataLoader as GeoLoader

import pgcn_experiment as E
from pgcn_core import DEAPDataset

warnings.filterwarnings("ignore")

BIN = ["val", "aro", "dom", "lik"]
ALL = BIN + ["quad"]
NCLS = {"val": 2, "aro": 2, "dom": 2, "lik": 2, "quad": 4}


# ─── helpers ──────────────────────────────────────────────────────────────────

def list_units(ckpt_dir: Path, mode: str):
    """Subjects (within) or folds (loso) that have a saved checkpoint."""
    units = []
    for d in sorted(ckpt_dir.iterdir()):
        if not d.is_dir() or not (d / "best.pt").exists():
            continue
        if mode == "within" and d.name.startswith("s"):
            units.append((d.name, d / "best.pt"))
        elif mode == "loso" and (d.name.startswith("fold_") or d.name.startswith("s")):
            units.append((d.name.replace("fold_", ""), d / "best.pt"))
    return units


_LABEL_CACHE = {}


def trial_label_table(data_dir: Path, subj: str, trials):
    """Trial-level labels [n_trials, 5]; one window per trial is enough."""
    key = (str(data_dir), subj, tuple(trials))
    if key in _LABEL_CACHE:
        return _LABEL_CACHE[key]
    rows = []
    for t in trials:
        g = torch.load(data_dir / subj / f"trial_{t:02d}" / "win_00.pt",
                       map_location="cpu", weights_only=False)
        rows.append(g.y.view(-1).numpy())
    out = np.stack(rows) if rows else np.zeros((0, 5), dtype=int)
    _LABEL_CACHE[key] = out
    return out


def majority_labels(data_dir: Path, subjects):
    """Majority class per task over the TRAINING trials of `subjects`."""
    ys = []
    for s in subjects:
        meta = json.load(open(data_dir / s / "meta.json"))
        ys.append(trial_label_table(data_dir, s, meta["train_trials"]))
    Y = np.concatenate(ys)
    maj = {}
    for i, t in enumerate(ALL):
        counts = np.bincount(Y[:, i].astype(int), minlength=NCLS[t])
        maj[t] = int(np.argmax(counts))          # ties resolve to the lower class
    return maj


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    P = {t: [] for t in ALL}
    Y, T = [], []
    for b in loader:
        b = b.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            out = model(b)
        for t in ALL:
            P[t].append(torch.softmax(out[t].float(), dim=-1).cpu().numpy())
        Y.append(b.y.view(-1, 5).cpu().numpy())
        T.append(b.trial_idx.view(-1).cpu().numpy())
    return ({t: np.concatenate(v) for t, v in P.items()},
            np.concatenate(Y).astype(int), np.concatenate(T).astype(int))


def score(y_true, y_pred, task):
    labels = list(range(NCLS[task]))
    r = {
        "macro_f1": f1_score(y_true, y_pred, average="macro",
                             labels=labels, zero_division=0),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
        "n_true_classes": int(len(np.unique(y_true))),
        "n": int(len(y_true)),
    }
    if task in BIN:
        r["f1_pos"] = f1_score(y_true, y_pred, average="binary", zero_division=0)
        r["pred_pos"] = float(np.mean(y_pred == 1))
    return r


def to_levels(probs, Y, T):
    """Return {level: {task: (y_true, y_pred)}} for window and trial levels."""
    levels = {"window": {}, "trial_mean": {}, "trial_vote": {}}
    trials = np.unique(T)
    for i, t in enumerate(ALL):
        p = probs[t]
        levels["window"][t] = (Y[:, i], p.argmax(1))
        yt, ym, yv = [], [], []
        for tr in trials:
            m = T == tr
            yt.append(np.bincount(Y[m, i]).argmax())
            ym.append(p[m].mean(0).argmax())
            yv.append(np.bincount(p[m].argmax(1), minlength=NCLS[t]).argmax())
        levels["trial_mean"][t] = (np.array(yt), np.array(ym))
        levels["trial_vote"][t] = (np.array(yt), np.array(yv))
    return levels


# ─── main ─────────────────────────────────────────────────────────────────────

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--ckpt_dir", required=True)
    ap.add_argument("--mode", choices=["within", "loso"], required=True)
    ap.add_argument("--fusion", choices=["none", "early", "late"], default="none")
    ap.add_argument("--ablation", default="none",
                    choices=["none", "flat_gcn", "no_band_attn",
                             "struct_only", "no_hjorth"])
    ap.add_argument("--out_prefix", required=True)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--bs", type=int, default=64)
    a = ap.parse_args()

    data_dir, ckpt_dir = Path(a.data_dir), Path(a.ckpt_dir)
    out_prefix = Path(a.out_prefix)
    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    device = torch.device(a.device)
    no_hjorth = a.ablation == "no_hjorth"
    all_subjects = E.get_available_subjects(str(data_dir))

    units = list_units(ckpt_dir, a.mode)
    if not units:
        raise SystemExit(f"No best.pt checkpoints found under {ckpt_dir}")
    print(f"{len(units)} checkpoints  mode={a.mode} fusion={a.fusion} "
          f"ablation={a.ablation}  device={device}")

    rows, pooled, saved = [], {}, {}
    for subj, ckpt in units:
        split = "test" if a.mode == "within" else "all"
        ds = DEAPDataset(str(data_dir), [subj], split, no_hjorth)
        if len(ds) == 0:
            print(f"  [SKIP] {subj}: no test windows")
            continue
        dl = GeoLoader(ds, batch_size=a.bs, shuffle=False, num_workers=2)

        model = E.build_model(a.mode, a.fusion, a.ablation).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        probs, Y, T = predict(model, dl, device)
        saved[subj] = {"Y": Y, "T": T, **{f"p_{t}": probs[t] for t in ALL}}

        train_subjects = [subj] if a.mode == "within" else \
                         [s for s in all_subjects if s != subj]
        maj = majority_labels(data_dir, train_subjects)

        for level, by_task in to_levels(probs, Y, T).items():
            for t, (yt, yp) in by_task.items():
                yb = np.full_like(yt, maj[t])
                for system, pred in (("model", yp), ("majority", yb)):
                    r = score(yt, pred, t)
                    rows.append({"subject": subj, "level": level, "task": t,
                                 "system": system, **r})
                    key = (level, t, system)
                    pooled.setdefault(key, ([], []))
                    pooled[key][0].append(yt)
                    pooled[key][1].append(pred)

        m = [r for r in rows if r["subject"] == subj and r["level"] == "window"
             and r["system"] == "model" and r["task"] in BIN]

        # Sanity check: the window-level positive-class F1 must reproduce the
        # val_f1/aro_f1/... saved in metrics.json next to the checkpoint.
        # A mismatch means the wrong checkpoint, data folder or config.
        check = ""
        mpath = ckpt.parent / "metrics.json"
        if mpath.exists():
            old = json.load(open(mpath))
            diffs = [abs(r["f1_pos"] - old[f"{r['task']}_f1"])
                     for r in m if f"{r['task']}_f1" in old]
            if diffs:
                worst = max(diffs)
                check = f"  match_metrics_json={'OK' if worst < 0.01 else 'MISMATCH'}({worst:.3f})"
                for r in rows:
                    if r["subject"] == subj:
                        r["max_diff_vs_metrics_json"] = worst

        print(f"  {subj}  window pred_pos=" +
              " ".join(f"{r['task']}:{r['pred_pos']:.2f}" for r in m) +
              "  bal_acc=" + " ".join(f"{r['task']}:{r['bal_acc']:.2f}" for r in m) +
              check)

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(f"{out_prefix}_per_subject.csv", index=False, float_format="%.4f")
    np.savez_compressed(f"{out_prefix}_window_probs.npz",
                        **{f"{s}__{k}": v for s, d in saved.items() for k, v in d.items()})

    # Pooled metrics: concatenate all subjects' predictions, then score once.
    # Use these for within-subject, where 6 test trials per subject make
    # per-subject macro-F1 unstable or undefined.
    prow = []
    for (level, t, system), (yts, yps) in pooled.items():
        r = score(np.concatenate(yts), np.concatenate(yps), t)
        prow.append({"level": level, "task": t, "system": system, **r})
    pdf = pd.DataFrame(prow)
    pdf.to_csv(f"{out_prefix}_pooled.csv", index=False, float_format="%.4f")

    cols = ["f1_pos", "macro_f1", "bal_acc", "pred_pos"]
    summ = (df.groupby(["level", "task", "system"])[cols]
              .agg(["mean", "std"]).round(3))
    summ.to_csv(f"{out_prefix}_summary.csv")
    with pd.option_context("display.width", 200, "display.max_rows", 200):
        print("\nMean across subjects (window level):")
        print(summ.loc["window"])
        print("\nMean across subjects (trial_mean level):")
        print(summ.loc["trial_mean"])
    print(f"\nSaved {out_prefix}_per_subject.csv, _pooled.csv, _summary.csv, "
          f"_window_probs.npz")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/stats_analysis.py
"""
stats_analysis.py
=================
Statistics on the re-scored checkpoints (output of reevaluate_checkpoints.py).
CPU only, no PyTorch needed.

Three families of paired, two-sided Wilcoxon signed-rank tests over subjects,
each corrected with Benjamini-Hochberg FDR inside its own family:

  F1  model vs majority-class baseline       (every config x task)
  F2  fusion vs EEG-only, matched subjects   (s01-s22, within and LOSO)
  F3  ablation vs full PGCN                  (within and LOSO)

Primary metric: balanced accuracy at window level. The same tests are repeated
for macro-F1 and at trial level (mean of the 29 window probabilities).

Usage:
  python stats_analysis.py --in_dir /kaggle/working/reeval_bs --out_dir /kaggle/working/stats
"""

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

TASKS = ["val", "aro", "dom", "lik", "quad"]
TASK_NAME = {"val": "Valence", "aro": "Arousal", "dom": "Dominance",
             "lik": "Liking", "quad": "Quadrant"}
ABL_ALIAS = {"no_band": "no_band_attn"}          # folder names differ by protocol


def bh(p):
    """Benjamini-Hochberg adjusted p-values (NaNs kept as NaN)."""
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    m = ok.sum()
    if m == 0:
        return out
    order = np.argsort(p[ok])
    ranked = p[ok][order] * m / np.arange(1, m + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.empty(m)
    adj[order] = np.minimum(ranked, 1.0)
    out[ok] = adj
    return out


def boot_ci(d, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    if len(d) < 2:
        return np.nan, np.nan
    idx = rng.integers(0, len(d), (n, len(d)))
    m = d[idx].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def paired_test(a, b):
    """Wilcoxon a vs b. Returns dict with means, diff, CI, p, effect size."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    nz = int(np.sum(np.abs(d) > 1e-12))
    if nz == 0:
        p, rbc = 1.0, 0.0
    else:
        try:
            p = float(wilcoxon(a, b, zero_method="wilcox",
                               alternative="two-sided").pvalue)
        except ValueError:
            p = 1.0
        dn = d[np.abs(d) > 1e-12]
        ranks = pd.Series(np.abs(dn)).rank().values
        rbc = float((ranks[dn > 0].sum() - ranks[dn < 0].sum()) / ranks.sum())
    lo, hi = boot_ci(d)
    return {"n": len(d), "n_nonzero": nz, "mean_a": a.mean(), "mean_b": b.mean(),
            "mean_diff": d.mean(), "ci_lo": lo, "ci_hi": hi,
            "p": p, "rank_biserial": rbc}


def load(in_dir: Path):
    frames = []
    for f in sorted(in_dir.glob("*_per_subject.csv")):
        name = f.name.replace("_per_subject.csv", "")
        mode, rest = name.split("_", 1)
        df = pd.read_csv(f)
        df["config"] = name
        df["mode"] = mode
        if rest in ("early", "late"):
            df["fusion"], df["ablation"] = rest, "none"
        else:
            df["fusion"], df["ablation"] = "none", ABL_ALIAS.get(rest, rest)
        frames.append(df)
    if not frames:
        raise SystemExit(f"No *_per_subject.csv in {in_dir}")
    return pd.concat(frames, ignore_index=True)


def pick(df, config, level, task, system, metric):
    s = df[(df.config == config) & (df.level == level) &
           (df.task == task) & (df.system == system)]
    return s.set_index("subject")[metric]


def family_vs_majority(df, level, metric):
    rows = []
    for cfg in sorted(df.config.unique()):
        for t in TASKS:
            m = pick(df, cfg, level, t, "model", metric)
            b = pick(df, cfg, level, t, "majority", metric)
            subs = m.index.intersection(b.index)
            if len(subs) == 0:
                continue
            r = paired_test(m[subs], b[subs])
            rows.append({"config": cfg, "task": t, **r})
    out = pd.DataFrame(rows)
    out["p_bh"] = bh(out.p)
    return out


def family_pairs(df, pairs, level, metric, matched_to=None):
    rows = []
    for a_cfg, b_cfg in pairs:
        if a_cfg not in df.config.values or b_cfg not in df.config.values:
            continue
        for t in TASKS:
            a = pick(df, a_cfg, level, t, "model", metric)
            b = pick(df, b_cfg, level, t, "model", metric)
            subs = a.index.intersection(b.index)
            if matched_to is not None:
                subs = subs.intersection(matched_to)
            r = paired_test(a[sorted(subs)], b[sorted(subs)])
            rows.append({"a": a_cfg, "b": b_cfg, "task": t, **r})
    out = pd.DataFrame(rows)
    if len(out):
        out["p_bh"] = bh(out.p)
    return out


def descriptive(df):
    cols = ["bal_acc", "macro_f1", "f1_pos", "pred_pos"]
    g = (df.groupby(["config", "level", "task", "system"])[cols]
           .agg(["mean", "std"]))
    g.columns = [f"{a}_{b}" for a, b in g.columns]
    return g.reset_index()


def fmt(x, nd=3):
    return "" if pd.isna(x) else f"{x:.{nd}f}"


def stars(p):
    if pd.isna(p):
        return ""
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    a = ap.parse_args()
    in_dir, out_dir = Path(a.in_dir), Path(a.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = load(in_dir)
    print(f"Loaded {df.config.nunique()} configs: {sorted(df.config.unique())}")

    if "max_diff_vs_metrics_json" in df.columns:
        md = df.groupby("config")["max_diff_vs_metrics_json"].max()
        print("\nLargest difference vs saved metrics.json per config "
              "(should be < 0.01 once batch sizes match training):")
        print(md.round(3).to_string())

    descriptive(df).to_csv(out_dir / "descriptive.csv", index=False,
                           float_format="%.4f")

    fusion_subs = pd.Index(sorted(df[df.fusion != "none"].subject.unique()))
    fusion_pairs = [("within_early", "within_none"), ("within_late", "within_none"),
                    ("within_late", "within_early"),
                    ("loso_early", "loso_none"), ("loso_late", "loso_none")]
    abl_pairs = []
    for mode in ("within", "loso"):
        for c in sorted(df[(df["mode"] == mode) & (df.fusion == "none")
                           & (df.ablation != "none")].config.unique()):
            abl_pairs.append((c, f"{mode}_none"))

    report = []
    for level in ("window", "trial_mean"):
        for metric in ("bal_acc", "macro_f1"):
            tag = f"{level}_{metric}"
            f1 = family_vs_majority(df, level, metric)
            f2 = family_pairs(df, fusion_pairs, level, metric, matched_to=fusion_subs)
            f3 = family_pairs(df, abl_pairs, level, metric)
            f1.to_csv(out_dir / f"F1_vs_majority_{tag}.csv", index=False, float_format="%.4f")
            f2.to_csv(out_dir / f"F2_fusion_vs_eeg_{tag}.csv", index=False, float_format="%.4f")
            f3.to_csv(out_dir / f"F3_ablation_{tag}.csv", index=False, float_format="%.4f")
            report.append((tag, f1, f2, f3))

    # pooled within-subject metrics (6 test trials per subject is too few alone)
    pooled = []
    for f in sorted(in_dir.glob("within_*_pooled.csv")):
        p = pd.read_csv(f)
        p["config"] = f.name.replace("_pooled.csv", "")
        pooled.append(p)
    if pooled:
        pooled = pd.concat(pooled)
        pooled.to_csv(out_dir / "pooled_within.csv", index=False, float_format="%.4f")

    # ---------- readable report (primary analysis) ----------
    lines = []
    for tag, f1, f2, f3 in report:
        lines.append(f"\n{'=' * 78}\n  {tag.upper()}\n{'=' * 78}")
        lines.append("\nF1  model vs majority baseline (Wilcoxon, BH over all configs x tasks)")
        lines.append(f"  {'config':22s}{'task':6s}{'n':>4s}{'model':>8s}{'major':>8s}"
                     f"{'diff':>8s}{'95% CI':>17s}{'p_BH':>9s}")
        for _, r in f1.iterrows():
            lines.append(f"  {r.config:22s}{r.task:6s}{r.n:4d}{fmt(r.mean_a):>8s}"
                         f"{fmt(r.mean_b):>8s}{fmt(r.mean_diff):>8s}"
                         f"  [{fmt(r.ci_lo)},{fmt(r.ci_hi)}]{fmt(r.p_bh, 4):>9s} {stars(r.p_bh)}")
        lines.append("\nF2  fusion vs EEG-only, matched subjects (BH over this family)")
        for _, r in f2.iterrows():
            lines.append(f"  {r.a:14s} vs {r.b:14s}{r.task:6s}{r.n:4d}{fmt(r.mean_a):>8s}"
                         f"{fmt(r.mean_b):>8s}{fmt(r.mean_diff):>8s}"
                         f"  [{fmt(r.ci_lo)},{fmt(r.ci_hi)}]{fmt(r.p_bh, 4):>9s} {stars(r.p_bh)}")
        lines.append("\nF3  ablation vs full PGCN (BH over this family)")
        for _, r in f3.iterrows():
            lines.append(f"  {r.a:22s} vs {r.b:12s}{r.task:6s}{r.n:4d}{fmt(r.mean_a):>8s}"
                         f"{fmt(r.mean_b):>8s}{fmt(r.mean_diff):>8s}"
                         f"  [{fmt(r.ci_lo)},{fmt(r.ci_hi)}]{fmt(r.p_bh, 4):>9s} {stars(r.p_bh)}")
    text = "\n".join(lines)
    (out_dir / "report.txt").write_text(text)
    print(text)
    print(f"\nSaved all tables to {out_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/check_labels.py
"""
check_labels.py
===============
Verifies the emotion labels used in training against the official DEAP ratings.

Why: some redistributed copies of DEAP have valence/arousal ratings reflected
on part of the trials. If your copy is one of them, every result changes.

Checks, per participant:
  A. DEAP .dat labels  vs  official participant_ratings (matched by Experiment_id)
  B. EEG preprocessed labels (win_00.pt y)     vs  labels derived from the .dat
  C. Fusion preprocessed labels (win_00.pt y)  vs  labels derived from the .dat
  D. EEG and fusion datasets use the same train/val/test trials

Usage:
  python check_labels.py --dat_dir <folder with s01.dat ...> \
      --eeg_dir <pgcn-deap-preprocessed> --fus_dir <preprocessed_fusion> \
      --ratings <participant_ratings.csv or .xls>   (optional but strongly advised)
"""

import argparse
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch

THR = 5.0                       # same threshold as preprocessing (rating > 5 = high)
COLS = ["Valence", "Arousal", "Dominance", "Liking"]


def binarize(r):
    r = np.asarray(r, float)
    v, a, d, l = (r[:, i] > THR for i in range(4))
    return np.stack([v, a, d, l, v * 2 + a], 1).astype(int)


def load_dat(dat_dir, subj):
    with open(Path(dat_dir) / f"{subj}.dat", "rb") as f:
        return pickle.load(f, encoding="latin1")["labels"].astype(float)   # (40, 4)


def load_official(path):
    p = Path(path)
    df = pd.read_excel(p) if p.suffix in (".xls", ".xlsx") else pd.read_csv(p)
    df.columns = [c.strip() for c in df.columns]
    low = {c.lower(): c for c in df.columns}
    need = ["participant_id", "experiment_id", "trial"] + [c.lower() for c in COLS]
    miss = [n for n in need if n not in low]
    if miss:
        raise SystemExit(f"Ratings file is missing columns {miss}. Found: {list(df.columns)}")
    return df.rename(columns={low[n]: n for n in need})


def preproc_labels(root, subj):
    """Binary labels [40, 5] stored in win_00.pt of each trial, plus meta."""
    sd = Path(root) / subj
    meta = json.load(open(sd / "meta.json"))
    ys = []
    for t in range(40):
        g = torch.load(sd / f"trial_{t:02d}" / "win_00.pt",
                       map_location="cpu", weights_only=False)
        ys.append(g.y.view(-1).numpy())
    return np.stack(ys).astype(int), meta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dat_dir", required=True)
    ap.add_argument("--eeg_dir", required=True)
    ap.add_argument("--fus_dir", required=True)
    ap.add_argument("--ratings", default=None)
    ap.add_argument("--out", default="label_check.csv")
    a = ap.parse_args()

    off = load_official(a.ratings) if a.ratings else None
    if off is None:
        print("!! No official ratings file given: check A is skipped.\n"
              "   Download metadata_csv.zip (participant_ratings.csv) from the DEAP\n"
              "   website, upload it as a Kaggle dataset, and pass --ratings.\n")

    rows = []
    for i in range(1, 33):
        s = f"s{i:02d}"
        if not (Path(a.dat_dir) / f"{s}.dat").exists():
            print(f"  {s}: .dat not found, skipped"); continue
        dat = load_dat(a.dat_dir, s)
        dat_bin = binarize(dat)
        r = {"subject": s}

        if off is not None:
            o = off[off.participant_id == i]
            o_by_exp = o.set_index("experiment_id").sort_index()
            o_by_trial = o.set_index("trial").sort_index()
            ref_exp = o_by_exp.loc[range(1, 41), [c.lower() for c in COLS]].values
            ref_trial = o_by_trial.loc[range(1, 41), [c.lower() for c in COLS]].values
            mism = np.any(np.abs(dat - ref_exp) > 0.01, axis=1)
            r["A_trials_rating_mismatch"] = int(mism.sum())
            r["A_valence_label_flips"] = int((binarize(ref_exp)[:, 0] != dat_bin[:, 0]).sum())
            r["A_arousal_label_flips"] = int((binarize(ref_exp)[:, 1] != dat_bin[:, 1]).sum())
            # diagnostic: does the .dat follow presentation order instead?
            r["A_mismatch_if_presentation_order"] = int(
                np.any(np.abs(dat - ref_trial) > 0.01, axis=1).sum())

        for tag, root in (("B_eeg", a.eeg_dir), ("C_fus", a.fus_dir)):
            if not (Path(root) / s / "meta.json").exists():
                r[f"{tag}_label_mismatch"] = None
                continue
            y, meta = preproc_labels(root, s)
            r[f"{tag}_label_mismatch"] = int(np.any(y != dat_bin, axis=1).sum())
            r[f"{tag}_meta"] = meta

        me, mf = r.pop("B_eeg_meta", None), r.pop("C_fus_meta", None)
        if me is not None and mf is not None:
            r["D_same_splits"] = all(sorted(me[k]) == sorted(mf[k])
                                     for k in ("train_trials", "val_trials", "test_trials"))
        rows.append(r)
        print("  " + "  ".join(f"{k}={v}" for k, v in r.items()), flush=True)

    df = pd.DataFrame(rows)
    df.to_csv(a.out, index=False)

    print("\n" + "=" * 70)
    problems = []
    for col in df.columns:
        if col.endswith("_mismatch") or col.endswith("_flips"):
            if col == "A_mismatch_if_presentation_order":
                continue
            bad = df[df[col].fillna(0) > 0]
            if len(bad):
                problems.append(f"{col}: {len(bad)} subjects, {int(bad[col].sum())} trials")
    if "D_same_splits" in df and (df.D_same_splits == False).any():
        problems.append(f"D_same_splits False for {list(df[df.D_same_splits == False].subject)}")
    if problems:
        print("  LABEL PROBLEMS FOUND. Do not use the results until this is fixed:")
        for p in problems:
            print("   -", p)
    else:
        print("  All checks passed" + ("" if off is not None
              else " (except A, which needs the official ratings file)"))
    print(f"  Table saved to {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/probe_analysis.py
"""
probe_analysis.py
=================
Linear probes on the frozen ViT-B/16 trial embeddings (R2.2, no network training).

For the 22 subjects with real video, one 768-d embedding per trial.
Two embedding versions:
  raw      : as used by the fusion models
  centred  : minus that subject's mean embedding over its own TRAINING trials
             (label-free; the same information EEG normalisation already uses)

Probes (logistic regression, class-balanced, standardised inputs):
  P1 identity  : which of the 22 subjects? (5-fold CV over trials; chance 1/22)
  P2 seen-subject / unseen-trial : train on train+val trials of all 22 subjects,
     test on their test trials (the within-subject split)
  P3 unseen-subject (LOSO) : train on 21 subjects, test on all 40 trials of the
     held-out subject
For P2 and P3: balanced accuracy per subject and pooled, a majority baseline,
and a label-permutation null (labels shuffled within each training subject).

Optional drift check (needs official participant_ratings.csv): do embeddings
encode presentation position in the session, and do labels drift with it?
If both hold, part of any within-subject video gain may reflect session drift.

Usage:
  python probe_analysis.py --fus_dir <preprocessed_fusion> --out_dir probes \
      [--ratings participant_ratings.csv] [--n_perm 100] [--n_jobs -1]
"""

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

TASKS = {"val": 0, "aro": 1, "dom": 2, "lik": 3, "quad": 4}


# ─── data ─────────────────────────────────────────────────────────────────────

def load_embeddings(fus_dir):
    import torch
    root = Path(fus_dir)
    X, Y, S, T, split = [], [], [], [], []
    for i in range(1, 33):
        s = f"s{i:02d}"
        mp = root / s / "meta.json"
        if not mp.exists():
            continue
        meta = json.load(open(mp))
        if not meta.get("has_real_video", False):
            continue
        sp = {t: "train" for t in meta["train_trials"]}
        sp.update({t: "val" for t in meta["val_trials"]})
        sp.update({t: "test" for t in meta["test_trials"]})
        for t in range(40):
            g = torch.load(root / s / f"trial_{t:02d}" / "win_00.pt",
                           map_location="cpu", weights_only=False)
            X.append(g.video.view(-1).float().numpy())
            Y.append(g.y.view(-1).numpy())
            S.append(s); T.append(t); split.append(sp[t])
    return (np.stack(X), np.stack(Y).astype(int), np.array(S),
            np.array(T), np.array(split))


def centre(X, S, split):
    """Subtract each subject's mean over its own training trials."""
    Xc = X.copy()
    for s in np.unique(S):
        m = S == s
        Xc[m] -= X[m & (split == "train")].mean(0)
    return Xc


def clf():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=1.0, max_iter=3000,
                                            class_weight="balanced"))


# ─── probes ───────────────────────────────────────────────────────────────────

def identity_probe(X, S, seed=0):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    pred = cross_val_predict(clf(), X, S, cv=cv)
    return float(np.mean(pred == S))


def per_subject_bacc(y, p, S):
    vals = []
    for s in np.unique(S):
        m = S == s
        if len(np.unique(y[m])) < 2:        # balanced accuracy undefined-ish
            vals.append(np.nan); continue
        vals.append(balanced_accuracy_score(y[m], p[m]))
    return float(np.nanmean(vals)), int(np.sum(~np.isnan(vals)))


def shuffle_within(y, S, rng):
    y = y.copy()
    for s in np.unique(S):
        m = np.where(S == s)[0]
        y[m] = y[rng.permutation(m)]
    return y


def run_seen(X, y, S, split, y_train=None):
    tr, te = split != "test", split == "test"
    yt = y if y_train is None else y_train
    p = clf().fit(X[tr], yt[tr]).predict(X[te])
    return p, y[te], S[te]


def run_loso(X, y, S, y_train=None):
    yt = y if y_train is None else y_train
    P = np.empty_like(y)
    for s in np.unique(S):
        te = S == s
        P[te] = clf().fit(X[~te], yt[~te]).predict(X[te])
    return P, y, S


def majority_pred(y, S, split, mode):
    if mode == "seen":
        tr, te = split != "test", split == "test"
        p = np.full(te.sum(), np.bincount(y[tr]).argmax())
        return p, y[te], S[te]
    P = np.empty_like(y)
    for s in np.unique(S):
        te = S == s
        P[te] = np.bincount(y[~te]).argmax()
    return P, y, S


def evaluate(p, y, S):
    ps, n = per_subject_bacc(y, p, S)
    return {"bacc_per_subject": ps, "n_subjects_scored": n,
            "bacc_pooled": float(balanced_accuracy_score(y, p))}


def perm_null(X, y, S, split, mode, n_perm, n_jobs, seed=0):
    def one(k):
        rng = np.random.default_rng(seed + k)
        ys = shuffle_within(y, S, rng)
        if mode == "seen":
            p, yy, ss = run_seen(X, y, S, split, y_train=ys)
        else:
            p, yy, ss = run_loso(X, y, S, y_train=ys)
        return balanced_accuracy_score(yy, p)
    return np.array(Parallel(n_jobs=n_jobs)(delayed(one)(k) for k in range(n_perm)))


# ─── drift check ──────────────────────────────────────────────────────────────

def drift_check(X, Y, S, T, ratings_path):
    p = Path(ratings_path)
    r = pd.read_excel(p) if p.suffix in (".xls", ".xlsx") else pd.read_csv(p)
    r.columns = [c.strip().lower() for c in r.columns]
    rows = []
    for s in np.unique(S):
        pid = int(s[1:])
        o = r[r.participant_id == pid].set_index("experiment_id")["trial"]
        m = S == s
        pos = o.loc[T[m] + 1].values.astype(float)        # trial index t = video t+1
        Xs = X[m]

        def cv_rho(target, seed):
            model = make_pipeline(StandardScaler(),
                                  RidgeCV(alphas=np.logspace(-1, 4, 12)))
            pred = cross_val_predict(model, Xs, target,
                                     cv=KFold(5, shuffle=True, random_state=seed))
            return spearmanr(pred, target).correlation

        rho_emb = cv_rho(pos, 0)
        # cross-validated regression is biased below 0 under the null, so the
        # reference is the same statistic with presentation positions shuffled
        rng = np.random.default_rng(int(s[1:]))
        null = [cv_rho(rng.permutation(pos), k) for k in range(20)]
        row = {"subject": s, "rho_embedding_predicts_position": rho_emb,
               "rho_null_mean": float(np.mean(null)),
               "rho_null_95": float(np.percentile(null, 95))}
        for t, k in TASKS.items():
            if t == "quad":
                continue
            yv = Y[m, k]
            row[f"rho_{t}_label_vs_position"] = (spearmanr(yv, pos).correlation
                                                 if len(np.unique(yv)) > 1 else np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


# ─── main ─────────────────────────────────────────────────────────────────────

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--fus_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--ratings", default=None)
    ap.add_argument("--n_perm", type=int, default=100)
    ap.add_argument("--n_jobs", type=int, default=-1)
    a = ap.parse_args()
    out = Path(a.out_dir); out.mkdir(parents=True, exist_ok=True)

    X, Y, S, T, split = load_embeddings(a.fus_dir)
    print(f"Embeddings: {X.shape}  subjects={len(np.unique(S))}")
    versions = {"raw": X, "centred": centre(X, S, split)}

    # P1 identity
    rows = [{"version": v, "identity_acc": identity_probe(Xv, S),
             "chance": 1 / len(np.unique(S))} for v, Xv in versions.items()]
    pd.DataFrame(rows).to_csv(out / "P1_identity.csv", index=False, float_format="%.4f")
    print("\nP1 identity probe"); print(pd.DataFrame(rows).round(3).to_string(index=False))

    # P2 / P3 affect probes
    res = []
    for mode in ("seen", "loso"):
        for t, k in TASKS.items():
            y = Y[:, k]
            mp, my, ms = majority_pred(y, S, split, mode)
            base = evaluate(mp, my, ms)
            for v, Xv in versions.items():
                p, yy, ss = (run_seen(Xv, y, S, split) if mode == "seen"
                             else run_loso(Xv, y, S))
                r = {"probe": "P2_seen_subject" if mode == "seen" else "P3_unseen_subject",
                     "task": t, "version": v, **evaluate(p, yy, ss),
                     "majority_bacc_pooled": base["bacc_pooled"]}
                if t in ("val", "aro", "quad") and a.n_perm > 0:
                    null = perm_null(Xv, y, S, split, mode, a.n_perm, a.n_jobs)
                    r["perm_null_mean"] = float(null.mean())
                    r["perm_null_95"] = float(np.percentile(null, 95))
                    r["perm_p"] = float((1 + np.sum(null >= r["bacc_pooled"])) / (1 + len(null)))
                res.append(r)
                print(f"  {r['probe']:18s} {t:5s} {v:8s} pooled bacc={r['bacc_pooled']:.3f} "
                      f"per-subj={r['bacc_per_subject']:.3f} "
                      f"majority={r['majority_bacc_pooled']:.3f}"
                      + (f"  perm_p={r['perm_p']:.3f}" if "perm_p" in r else ""), flush=True)
    pd.DataFrame(res).to_csv(out / "P2_P3_affect_probes.csv", index=False, float_format="%.4f")

    # drift check
    if a.ratings:
        d = drift_check(versions["raw"], Y, S, T, a.ratings)
        d.to_csv(out / "drift_check.csv", index=False, float_format="%.4f")
        print("\nDrift check (Spearman, mean over subjects):")
        print(d.drop(columns="subject").mean().round(3).to_string())
        n_above = int((d.rho_embedding_predicts_position > d.rho_null_95).sum())
        print(f"  subjects whose embeddings predict session position above their "
              f"95% null: {n_above}/{len(d)}")
        print("  Compare rho_embedding_predicts_position with rho_null_mean, not with 0.\n"
              "  If embeddings track position AND labels also track position, session\n"
              "  drift could explain part of the within-subject video gain.")
    else:
        print("\n(Drift check skipped: pass --ratings participant_ratings.csv)")
    print(f"\nSaved to {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/video_shuffle_test.py
"""
video_shuffle_test.py
=====================
Test-time video manipulation on the trained WITHIN-SUBJECT fusion models
(R2.2, inference only, no training).

For each subject's 6 test trials the EEG input is unchanged and only the video
embedding is replaced:
  original      : the trial's own video embedding
  other_trial   : the embedding of a randomly chosen DIFFERENT trial of the same
                  subject (repeated n_rep times, results averaged)
  subject_mean  : the subject's mean embedding over its training trials
                  (same identity, no trial-specific information)

If the within-subject fusion gain depends on trial-specific video content,
other_trial and subject_mean should fall back towards EEG-only performance.

Batch size 32 matches training (BandAttn uses batch statistics).

Usage:
  python video_shuffle_test.py --fus_dir <preprocessed_fusion> \
      --ckpt_root <.../results_fusion> --out_dir shuffle [--n_rep 20]
"""

import argparse
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.metrics import balanced_accuracy_score, f1_score
from torch_geometric.loader import DataLoader as GeoLoader

import pgcn_experiment as E
from pgcn_core import DEAPDataset

warnings.filterwarnings("ignore")
TASKS = ["val", "aro", "dom", "lik", "quad"]
NCLS = {"val": 2, "aro": 2, "dom": 2, "lik": 2, "quad": 4}


def video_table(fus_dir, subj):
    """[40, 768] video embedding per trial (constant across windows)."""
    vids = []
    for t in range(40):
        g = torch.load(Path(fus_dir) / subj / f"trial_{t:02d}" / "win_00.pt",
                       map_location="cpu", weights_only=False)
        vids.append(g.video.view(-1).float())
    return torch.stack(vids)


@torch.no_grad()
def predict(model, loader, device, vid_for_trial):
    """vid_for_trial: tensor [40, 768] giving the video to feed for each trial."""
    model.eval()
    P = {t: [] for t in TASKS}
    Y, T = [], []
    vt = vid_for_trial.to(device)
    for b in loader:
        b = b.to(device, non_blocking=True)
        tidx = b.trial_idx.view(-1).long()
        b.video = vt[tidx]
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            out = model(b)
        for t in TASKS:
            P[t].append(torch.softmax(out[t].float(), -1).cpu().numpy())
        Y.append(b.y.view(-1, 5).cpu().numpy())
        T.append(tidx.cpu().numpy())
    return ({t: np.concatenate(v) for t, v in P.items()},
            np.concatenate(Y).astype(int), np.concatenate(T))


def scores(P, Y, T):
    r = {}
    trials = np.unique(T)
    for i, t in enumerate(TASKS):
        yw, pw = Y[:, i], P[t].argmax(1)
        r[f"{t}_bacc_window"] = balanced_accuracy_score(yw, pw)
        r[f"{t}_macroF1_window"] = f1_score(yw, pw, average="macro",
                                            labels=list(range(NCLS[t])), zero_division=0)
        yt = np.array([np.bincount(Y[T == k, i]).argmax() for k in trials])
        pt = np.array([P[t][T == k].mean(0).argmax() for k in trials])
        r[f"{t}_bacc_trial"] = balanced_accuracy_score(yt, pt)
    return r


def bh(p):
    p = np.asarray(p, float); m = len(p)
    o = np.argsort(p); r = p[o] * m / np.arange(1, m + 1)
    r = np.minimum.accumulate(r[::-1])[::-1]
    out = np.empty(m); out[o] = np.minimum(r, 1); return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--fus_dir", required=True)
    ap.add_argument("--ckpt_root", required=True,
                    help="folder containing within_early/ and within_late/")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--n_rep", type=int, default=20)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    out = Path(a.out_dir); out.mkdir(parents=True, exist_ok=True)
    device = torch.device(a.device)

    rows = []
    for fusion in ("early", "late"):
        cdir = Path(a.ckpt_root) / f"within_{fusion}"
        subs = sorted(d.name for d in cdir.iterdir()
                      if d.is_dir() and (d / "best.pt").exists())
        print(f"\nwithin_{fusion}: {len(subs)} checkpoints")
        for s in subs:
            meta = json.load(open(Path(a.fus_dir) / s / "meta.json"))
            ds = DEAPDataset(a.fus_dir, [s], "test")
            dl = GeoLoader(ds, batch_size=a.bs, shuffle=False, num_workers=2)
            model = E.build_model("within", fusion, "none").to(device)
            model.load_state_dict(torch.load(cdir / s / "best.pt",
                                             map_location=device, weights_only=True))
            V = video_table(a.fus_dir, s)
            rng = np.random.default_rng(int(s[1:]))

            # original
            r0 = scores(*predict(model, dl, device, V))
            rows.append({"fusion": fusion, "subject": s, "condition": "original", **r0})

            # subject mean over training trials
            vm = V[meta["train_trials"]].mean(0, keepdim=True).expand(40, -1)
            rm = scores(*predict(model, dl, device, vm))
            rows.append({"fusion": fusion, "subject": s, "condition": "subject_mean", **rm})

            # other trial of the same subject (never its own)
            reps = []
            for _ in range(a.n_rep):
                src = np.array([rng.choice([k for k in range(40) if k != t])
                                for t in range(40)])
                reps.append(scores(*predict(model, dl, device, V[src])))
            ro = pd.DataFrame(reps).mean().to_dict()
            rows.append({"fusion": fusion, "subject": s, "condition": "other_trial", **ro})

            print(f"  {s}  valence bacc: original={r0['val_bacc_window']:.2f} "
                  f"other_trial={ro['val_bacc_window']:.2f} "
                  f"subject_mean={rm['val_bacc_window']:.2f}", flush=True)
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(out / "video_shuffle_per_subject.csv", index=False, float_format="%.4f")

    # paired tests: original vs each manipulation
    tests = []
    for fusion in ("early", "late"):
        d = df[df.fusion == fusion]
        o = d[d.condition == "original"].set_index("subject")
        for cond in ("other_trial", "subject_mean"):
            c = d[d.condition == cond].set_index("subject").loc[o.index]
            for t in TASKS:
                for lvl in ("window", "trial"):
                    col = f"{t}_bacc_{lvl}"
                    diff = o[col].values - c[col].values
                    p = 1.0 if np.allclose(diff, 0) else wilcoxon(o[col], c[col]).pvalue
                    tests.append({"fusion": fusion, "manipulation": cond, "task": t,
                                  "level": lvl, "original": o[col].mean(),
                                  "manipulated": c[col].mean(),
                                  "drop": diff.mean(), "p": p})
    tt = pd.DataFrame(tests)
    tt["p_bh"] = bh(tt.p.values)
    tt.to_csv(out / "video_shuffle_tests.csv", index=False, float_format="%.4f")
    with pd.option_context("display.width", 200, "display.max_rows", 200):
        print("\nOriginal vs manipulated video (window-level balanced accuracy):")
        print(tt[tt.level == "window"].round(3).to_string(index=False))
    print(f"\nSaved to {out}")


if __name__ == "__main__":
    main()

In [ ]:
# 3. Find inputs, copy model code
import shutil, os, sys, subprocess
from pathlib import Path
IN = Path("/kaggle/input")

def find_one(pattern, desc, required=True):
    hits = sorted(IN.rglob(pattern))
    if not hits:
        if required: raise FileNotFoundError(f"{desc} not found ({pattern}). Add it as an input.")
        print(f"[missing] {desc}"); return None
    return hits[0]

BASE = find_one("all_results", "pgcn-all-results")
def find_data(keyword):
    for p in IN.rglob("meta.json"):
        if keyword in str(p) and p.parent.name == "s01":
            return str(p.parent.parent)
    raise FileNotFoundError(keyword)
EEG = find_data("pgcn-deap-preprocessed")
FUS = find_data("fusion")
DAT = str(find_one("s01.dat", "DEAP .dat files").parent)
RAT = find_one("participant_ratings.*", "official participant_ratings file", required=False)
for f in ["pgcn_experiment.py", "pgcn_model.py", "pgcn_core.py"]:
    shutil.copy(find_one(f, f), "/kaggle/working/")
sys.path.insert(0, "/kaggle/working")
print("BASE", BASE, "\nEEG ", EEG, "\nFUS ", FUS, "\nDAT ", DAT, "\nRAT ", RAT)

def run(args):
    """Run a script with live output; stop the notebook on error."""
    p = subprocess.Popen([sys.executable, "-u", *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, cwd="/kaggle/working")
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{args[0]} failed (see output above)")

## 4. Label check
Read the last lines. If it says **LABEL PROBLEMS FOUND**, stop and send me `label_check.csv`.

In [ ]:
args = ["check_labels.py", "--dat_dir", DAT, "--eeg_dir", EEG, "--fus_dir", FUS,
        "--out", "/kaggle/working/label_check.csv"]
if RAT: args += ["--ratings", str(RAT)]
run(args)

## 5. Re-score all checkpoints with the training batch sizes
Every subject should print `match_metrics_json=OK`.

In [ ]:
ABL = {"none": "none", "flat_gcn": "flat_gcn", "no_band": "no_band_attn",
       "no_band_attn": "no_band_attn", "no_hjorth": "no_hjorth", "struct_only": "struct_only"}
RUNS = []
for d in sorted(BASE.glob("results_*/*")):
    mode, rest = d.name.split("_", 1)
    if d.parent.name == "results_fusion":
        RUNS.append((d.name, d, FUS, mode, rest, "none"))
    else:
        RUNS.append((d.name, d, EEG, mode, "none", ABL[rest]))
OUT = "/kaggle/working/reeval_bs"; os.makedirs(OUT, exist_ok=True)
for name, ck, data, mode, fus, abl in sorted(RUNS, key=lambda r: r[3] != "within"):
    if os.path.exists(f"{OUT}/{name}_summary.csv"):
        print("[skip]", name); continue
    print(f"\n===== {name} =====", flush=True)
    run(["reevaluate_checkpoints.py", "--data_dir", data, "--ckpt_dir", str(ck),
         "--mode", mode, "--fusion", fus, "--ablation", abl,
         "--bs", "32" if mode == "within" else "16", "--out_prefix", f"{OUT}/{name}"])

## 6. Statistics

In [ ]:
run(["stats_analysis.py", "--in_dir", "/kaggle/working/reeval_bs",
     "--out_dir", "/kaggle/working/stats"])

## 7. ViT embedding probes (CPU, about 10-20 minutes)

In [ ]:
args = ["probe_analysis.py", "--fus_dir", FUS, "--out_dir", "/kaggle/working/probes",
        "--n_perm", "100", "--n_jobs", "-1"]
if RAT: args += ["--ratings", str(RAT)]
run(args)

## 8. Test-time video shuffle on the within-subject fusion models

In [ ]:
run(["video_shuffle_test.py", "--fus_dir", FUS, "--ckpt_root", str(BASE / "results_fusion"),
     "--out_dir", "/kaggle/working/shuffle", "--n_rep", "20"])

## 9. Download everything
Click the link, then upload the zip in our chat.

In [ ]:
import shutil
from IPython.display import FileLink
work = Path("/kaggle/working/revision_results"); work.mkdir(exist_ok=True)
for d in ["stats", "probes", "shuffle"]:
    if Path(f"/kaggle/working/{d}").exists():
        shutil.copytree(f"/kaggle/working/{d}", work / d, dirs_exist_ok=True)
(work / "reeval_bs").mkdir(exist_ok=True)
for f in Path("/kaggle/working/reeval_bs").glob("*.csv"):
    shutil.copy(f, work / "reeval_bs")
if Path("/kaggle/working/label_check.csv").exists():
    shutil.copy("/kaggle/working/label_check.csv", work)
shutil.make_archive("/kaggle/working/revision_results", "zip", work)
FileLink("revision_results.zip")